# NYC Taxi | Gold aggregation and Delta inspection

## Goal
Create a dashboard-ready day/hour table while separating volume counts from eligible-only distance and duration averages.

## Setup
Run notebook 02 first. Notebook 03 is read-only analysis and is optional for building Gold.

**Write behavior:** `CREATE OR REPLACE TABLE` fully refreshes the project Gold table from Silver. Existing Delta history is retained, so a rerun creates another version. Selected unchanged queries retain recorded output tables from the original export. Table-history/detail outputs were cleared because they included session and account metadata.

## Steps

### Create or refresh the Gold Delta table

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.nyc_taxi_gold_gun_saat_2025_01
USING DELTA
AS
SELECT
    CAST(tpep_pickup_datetime AS DATE) AS binis_tarihi,
    HOUR(tpep_pickup_datetime) AS binis_saati,

    COUNT(*) AS yolculuk_kaydi_sayisi,

    SUM(
        CASE WHEN olcum_analizine_uygun THEN 1 ELSE 0 END
    ) AS olcume_uygun_kayit_sayisi,

    AVG(
        CASE WHEN olcum_analizine_uygun THEN trip_distance END
    ) AS ortalama_mesafe_mil,

    AVG(
        CASE WHEN olcum_analizine_uygun THEN sure_dakika END
    ) AS ortalama_sure_dakika

FROM workspace.default.nyc_taxi_silver_2025_01
GROUP BY
    CAST(tpep_pickup_datetime AS DATE),
    HOUR(tpep_pickup_datetime);

### Reconcile the monthly totals

In [ ]:
%sql
SELECT
    COUNT(*) AS ozet_satir_sayisi,
    MIN(binis_tarihi) AS ilk_tarih,
    MAX(binis_tarihi) AS son_tarih,
    SUM(yolculuk_kaydi_sayisi) AS toplam_yolculuk_kaydi,
    SUM(olcume_uygun_kayit_sayisi) AS toplam_uygun_kayit
FROM workspace.default.nyc_taxi_gold_gun_saat_2025_01;

ozet_satir_sayisi,ilk_tarih,son_tarih,toplam_yolculuk_kaydi,toplam_uygun_kayit
744,2025-01-01,2025-01-31,3475204,3380592


### Inspect the first day/hour rows

In [ ]:
%sql
SELECT *
FROM workspace.default.nyc_taxi_gold_gun_saat_2025_01
ORDER BY binis_tarihi, binis_saati
LIMIT 24;

binis_tarihi,binis_saati,yolculuk_kaydi_sayisi,olcume_uygun_kayit_sayisi,ortalama_mesafe_mil,ortalama_sure_dakika
2025-01-01,0,7344,7247,3.329176210845868,16.58340232739988
2025-01-01,1,8468,8335,3.3108578284343064,16.028900219955986
2025-01-01,2,7257,7069,3.4901612675060116,15.025793370113645
2025-01-01,3,4915,4780,3.5266861924686204,13.938235704323546
2025-01-01,4,2918,2827,3.9476759816059457,13.914491215658584
2025-01-01,5,1429,1358,4.958748159057443,14.464310260186554
2025-01-01,6,1220,1185,6.126126582278486,15.071462728551358
2025-01-01,7,1116,1089,5.841331496786041,14.001040710131612
2025-01-01,8,1116,1090,5.459724770642198,13.73099388379205
2025-01-01,9,1524,1501,4.587948034643567,12.499478125693981


### Check Gold/Silver reconciliation and the day/hour grain

In [ ]:
%sql
WITH silver_counts AS (
    SELECT COUNT(*) AS total_records,
           SUM(CASE WHEN olcum_analizine_uygun THEN 1 ELSE 0 END) AS eligible_records
    FROM workspace.default.nyc_taxi_silver_2025_01
), gold_counts AS (
    SELECT COUNT(*) AS summary_rows,
           SUM(yolculuk_kaydi_sayisi) AS total_records,
           SUM(olcume_uygun_kayit_sayisi) AS eligible_records,
           COUNT(DISTINCT STRUCT(binis_tarihi, binis_saati)) AS unique_keys
    FROM workspace.default.nyc_taxi_gold_gun_saat_2025_01
)
SELECT
    g.summary_rows,
    g.summary_rows = g.unique_keys AS keys_are_unique,
    g.total_records = s.total_records AS volume_reconciles,
    g.eligible_records = s.eligible_records AS eligibility_reconciles
FROM gold_counts g CROSS JOIN silver_counts s;

### Inspect Delta history

In [ ]:
%sql
DESCRIBE HISTORY workspace.default.nyc_taxi_gold_gun_saat_2025_01;

### Inspect the current table details

In [ ]:
%sql
DESCRIBE DETAIL workspace.default.nyc_taxi_gold_gun_saat_2025_01;

### Read the first retained Gold snapshot

In [ ]:
%sql
SELECT
    COUNT(*) AS ozet_satir_sayisi,
    SUM(yolculuk_kaydi_sayisi) AS toplam_yolculuk_kaydi,
    SUM(olcume_uygun_kayit_sayisi) AS toplam_uygun_kayit
FROM workspace.default.nyc_taxi_gold_gun_saat_2025_01
VERSION AS OF 0;

## Checks and recorded baseline
The original recorded Gold snapshot has 744 unique day/hour groups, from 2025-01-01 through 2025-01-31. Total volume is 3,475,204 records; eligible volume is 3,380,592.

`VERSION AS OF 0` reads the first retained snapshot, which can differ from the current version after input changes. A version number is table-specific. Historical reads also depend on retained data files.

**Rollup rule:** sum count columns. For an overall eligible mean, weight each hourly mean by `olcume_uygun_kayit_sayisi` (using only non-null means), or calculate directly from Silver. An unweighted average of hourly averages is not the monthly mean.

## Next steps
Use `sql/dashboard_dataset.sql` in the Databricks dashboard SQL editor and follow `docs/DASHBOARD.md`. Notebook 05 demonstrates mutation and restore on a separate one-row table.